In [ ]:
# OPTIONAL: Install extras for tools/agents
# pip install -qU langchain langchain-community langchain-openai duckduckgo-search wikipedia numexpr python-dotenv


## Tutorial: Tools, Chains, Agents — Core Concepts (CRM Mini-App)

We’ll implement a tiny in-memory CRM to ground the concepts:
- **Tool**: `add_contact` and `get_contact` that mutate/read a store.
- **Chain**: LLM formatting/summarization of tool results.
- **Agent**: LLM decides whether to add or read and routes accordingly.

Goal: show how tools (actions), chains (deterministic formatting), and agents (routing) fit together in a practical example.


### Step 1: Imports and model
Small, deterministic config; keep creds outside the notebook.


In [ ]:
from getpass import getpass


# OpenRouter configuration
from getpass import getpass

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (hidden): ")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model="openai/gpt-4o-mini", temperature=0, seed=42, api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)


### Step 2: CRM Tools (in-memory)
We’ll create two tools over a simple in-memory store:
- `add_contact(name, email, phone)` → adds or updates a contact
- `get_contact(name)` → returns contact details or a not-found message


In [ ]:
from typing import Dict, Optional

crm_store: Dict[str, Dict[str, str]] = {}

def add_contact(name: str, email: Optional[str] = None, phone: Optional[str] = None) -> Dict[str, str]:
    """Add or update a contact in the in-memory store and return the record."""
    if not name or not isinstance(name, str):
        return {"status": "error", "message": "Name is required and must be a string."}
    record = crm_store.get(name.strip(), {})
    if email:
        record["email"] = email.strip()
    if phone:
        record["phone"] = phone.strip()
    record["name"] = name.strip()
    crm_store[name.strip()] = record
    return {"status": "ok", "message": "Contact saved.", "contact": record}

def get_contact(name: str) -> Dict[str, str]:
    """Retrieve a contact by name from the in-memory store."""
    if not name or not isinstance(name, str):
        return {"status": "error", "message": "Name is required and must be a string."}
    record = crm_store.get(name.strip())
    if record:
        return {"status": "ok", "contact": record}
    return {"status": "not_found", "message": f"No contact found for '{name.strip()}'"}

# Quick sanity check
print(add_contact("Alice", email="alice@example.com"))
print(get_contact("Alice"))


### Step 3: LLM Formatting Chain
We’ll use a deterministic chain to format contact data (or errors) from the tools into a concise response.


In [ ]:
format_prompt = PromptTemplate.from_template(
    """
    You are a helpful CRM assistant. Given the raw JSON-like dict below,
    produce a concise, user-facing one-liner. If status is ok and contact exists,
    return "Name — email, phone" (omit missing fields). If not_found or error,
    return the message as-is. Input:
    {tool_result}
    """
)

format_chain = LLMChain(llm=llm, prompt=format_prompt)

def format_tool_result(tool_result: dict) -> str:
    return format_chain.run({"tool_result": str(tool_result)})

# Sanity check formatting
print(format_tool_result(get_contact("Alice")))


### Step 4: Agent Router (choice: add vs read)
An agent chooses which tool to call based on the user's intent:
- If the user asks to save/add/update a contact → call `add_contact`
- If the user asks to look up/find/show a contact → call `get_contact`
We'll use a simple intent classifier and parameter extractor.


In [ ]:
from typing import Literal, Tuple, Dict
import re

Intent = Literal["add", "read", "unknown"]

intent_prompt = PromptTemplate.from_template(
    """
    Determine if the user wants to add/update a contact or read a contact.
    Reply with only one word: add, read, or unknown.
    User: {text}
    """
)
intent_chain = LLMChain(llm=llm, prompt=intent_prompt)

extract_add_prompt = PromptTemplate.from_template(
    """
    Extract fields for adding a contact from the text. Use JSON with keys name, email, phone.
    Missing fields should be null. Text: {text}
    """
)
extract_add_chain = LLMChain(llm=llm, prompt=extract_add_prompt)

extract_read_prompt = PromptTemplate.from_template(
    """
    Extract the contact name to look up. Reply with just the name string.
    Text: {text}
    """
)
extract_read_chain = LLMChain(llm=llm, prompt=extract_read_prompt)

In [ ]:

def classify_intent(text: str) -> Intent:
    out = intent_chain.run({"text": text}).strip().lower()
    if out.startswith("add"):
        return "add"
    if out.startswith("read") or out.startswith("lookup") or out.startswith("get"):
        return "read"
    return "unknown"

def parse_add_fields(text: str) -> Tuple[str, Dict[str, str]]:
    raw = extract_add_chain.run({"text": text})
    # Try to extract JSON-ish fields; fallback to regex if needed
    name = None
    email = None
    phone = None
    try:
        import json
        js = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
        name = js.get("name")
        email = js.get("email")
        phone = js.get("phone")
    except Exception:
        # naive heuristics
        m_name = re.search(r"name[:=]\s*([\w\s]+)", raw, re.I)
        if m_name:
            name = m_name.group(1).strip()
        m_email = re.search(r"[\w\.-]+@[\w\.-]+", raw)
        if m_email:
            email = m_email.group(0)
        m_phone = re.search(r"(\+?\d[\d\s\-]{6,}\d)", raw)
        if m_phone:
            phone = m_phone.group(1)
    return name, {"email": email, "phone": phone}

def parse_read_name(text: str) -> str:
    name = extract_read_chain.run({"text": text}).strip()
    # Clean quotes
    name = re.sub(r'^[\"\']|[\"\']$', "", name)
    return name

def crm_agent(user_input: str) -> str:
    intent = classify_intent(user_input)
    if intent == "add":
        name, fields = parse_add_fields(user_input)
        if not name:
            return "Please provide a contact name to add/update."
        result = add_contact(name=name, email=fields.get("email"), phone=fields.get("phone"))
        return format_tool_result(result)
    if intent == "read":
        name = parse_read_name(user_input)
        if not name:
            return "Please provide the contact name to look up."
        result = get_contact(name)
        return format_tool_result(result)
    return "I can add or look up contacts. Try: 'Add John Doe john@x.com +1 555 123 4567' or 'Find John Doe'."


In [ ]:
# Demos
print(crm_agent("Add Alice Cooper with email alice@co.com and phone +1 202 555 0142"))
print(crm_agent("Find Alice Cooper"))

### Step 5: When to use which?
- **Chain**: deterministic workflow, known steps. Faster, cheaper, easier to test.
- **Agent**: dynamic tasks, unknown number/order of steps, tool choice.
- Start with chains, graduate to agents when branching logic explodes.

You now have intuition + a tiny agent sketch. In later sections, we’ll wire full LangChain Tools and Agents.
